# Notebook 14: Sequential Multi-Agent Investment System
## Agent 1 → Agent 2 → Agent 3

This notebook chains all three investment agents into a single **Sequential
Multi-Agent System**. Each agent runs in order and passes its output to the next.

```
┌─────────────────────────────────────────────────────────────────────┐
│                   SEQUENTIAL MULTI-AGENT SYSTEM                      │
│                                                                       │
│  User Input                                                           │
│      │                                                                │
│      ▼                                                                │
│  ┌──────────┐   portfolio.json    ┌──────────┐  consolidated.json    │
│  │ AGENT 1  │ ──────────────────► │ AGENT 2  │ ─────────────────►   │
│  │ Designer │                     │Consolidat│        │              │
│  │          │                     │  or +    │        │              │
│  │ Profiles │                     │ Analyser │        ▼              │
│  │ investor │                     │          │  ┌──────────┐         │
│  │ builds   │                     │ Merges   │  │ AGENT 3  │         │
│  │portfolio │                     │ portfoli │  │ Educator │         │
│  └──────────┘                     └──────────┘  │          │         │
│                                                  │Explains  │         │
│                                                  │concepts  │         │
│                                                  └──────────┘         │
│                                                       │               │
│                                                       ▼               │
│                                               Final Response          │
└─────────────────────────────────────────────────────────────────────┘
```

## Routing Logic
The **Router** analyses each user query and decides which agent(s) to invoke:

| Query Type | Route |
|------------|-------|
| Build / design a portfolio | Agent 1 → Agent 2 → Agent 3 |
| Portfolio analytics / performance | Agent 2 → Agent 3 |
| Explain a concept | Agent 3 only |
| Mixed (profile + educate) | Agent 1 → Agent 3 |

## Test Queries (in order)
1. `"Help me build a long-term portfolio."`
2. `"I am 45, moderate risk, retiring in 15 years — what allocation should I use?"`
3. `"What is beta?"`
4. `"What is dollar-cost averaging?"`
5. `"How volatile is my portfolio?"`

## Installation

In [ ]:
%pip install langchain langchain-openai langchain-community langgraph \
             python-dotenv google-search-results \
             pandas openpyxl pypdf pillow --quiet
print("Packages installed")

## Imports & Configuration

In [ ]:
import json
import os
from pathlib import Path
from typing import Literal, TypedDict, Annotated
import operator

import pandas as pd
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

load_dotenv()
load_dotenv('../.env')

try:
    from ai_course_utils import load_llm_from_env, display_config
    USE_COURSE_UTILS = True
    display_config()
except ImportError:
    USE_COURSE_UTILS = False
    def load_llm_from_env(temperature: float = 0.0):
        model = os.getenv('LLM_MODEL', 'gpt-4o-mini')
        return ChatOpenAI(model=model, temperature=temperature)
    print(f"LLM_MODEL : {os.getenv('LLM_MODEL','gpt-4o-mini')}")
    print(f"OpenAI key: {'Set ✅' if os.getenv('OPENAI_API_KEY') else 'NOT SET ⚠️'}")
    print(f"Serper key: {'Set ✅' if os.getenv('SERPER_API_KEY') else 'NOT SET ⚠️'}")

print("Imports successful")

## File Paths & Shared State

In [ ]:
USER_ID             = "user1"
INPUT_DIR           = Path(f"../data/input/{USER_ID}")
OUTPUT_DIR          = Path(f"../data/outputs/{USER_ID}")
AGENT1_JSON         = Path("../data/outputs/portfolio.json")
CONSOLIDATED_JSON   = OUTPUT_DIR / "consolidated_portfolio.json"
CONSOLIDATED_EXCEL  = OUTPUT_DIR / "consolidated_portfolio.xlsx"
QUESTIONNAIRE_PATH  = Path("../data/Portfolio_Questionnaire.pdf")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Shared in-memory state passed between agents ──────────────────────────────
SHARED_STATE = {
    "investor_profile"   : None,   # set by Agent 1
    "portfolio"          : None,   # set by Agent 1 (dict)
    "consolidated"       : None,   # set by Agent 2 (list of holdings)
    "analytics"          : None,   # set by Agent 2 (analytics dict)
    "conversation_history": [],    # cross-agent context window
}

print(f"Input  : {INPUT_DIR}")
print(f"Output : {OUTPUT_DIR}")
print("Shared state initialised.")

## Pre-Loaded Sample Portfolio Data (from 5 PDFs)

The five sample portfolios from the uploaded PDFs are embedded here so the
system works out-of-the-box without requiring files on disk.

In [ ]:
SAMPLE_PORTFOLIOS = [
    # Portfolio 1 — Conservative IRA
    {"ticker":"BND",  "company_name":"Vanguard Total Bond Market ETF",          "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":620.45,"share_price":75.20,  "amount_usd":46679,  "gain_loss_usd": 1420.45,"gain_loss_pct": 3.13, "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"LQD",  "company_name":"iShares iBoxx Investment Grade Corp Bond", "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":330.12,"share_price":109.25, "amount_usd":36049,  "gain_loss_usd":-920.12, "gain_loss_pct":-2.49, "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"SCHP", "company_name":"Schwab US TIPS ETF",                       "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":480.00,"share_price":56.14,  "amount_usd":26947,  "gain_loss_usd":  347.00,"gain_loss_pct": 1.31, "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"VTI",  "company_name":"Vanguard Total Stock Market ETF",          "investment_type":"ETF",        "asset_class":"US Equity",    "shares":110.00,"share_price":335.40, "amount_usd":36894,  "gain_loss_usd": 4102.00,"gain_loss_pct":12.50, "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"VEA",  "company_name":"Vanguard FTSE Developed Markets ETF",      "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":290.00,"share_price":62.45,  "amount_usd":18110,  "gain_loss_usd":  980.12,"gain_loss_pct": 5.72, "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    {"ticker":"VWO",  "company_name":"Vanguard FTSE Emerging Markets ETF",       "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":200.00,"share_price":53.90,  "amount_usd":10780,  "gain_loss_usd": -510.34,"gain_loss_pct":-4.52, "source_file":"Portfolio1.pdf","source_account":"Conservative IRA"},
    # Portfolio 2 — Taxable Stock Heavy
    {"ticker":"AAPL", "company_name":"Apple Inc.",                               "investment_type":"Stock",      "asset_class":"US Equity",    "shares":320.51,"share_price":262.11, "amount_usd":84022,  "gain_loss_usd":14901.00,"gain_loss_pct":21.54, "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"MSFT", "company_name":"Microsoft Corp.",                          "investment_type":"Stock",      "asset_class":"US Equity",    "shares":190.00,"share_price":412.34, "amount_usd":78344,  "gain_loss_usd":20120.00,"gain_loss_pct":34.54, "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"NVDA", "company_name":"NVIDIA Corporation",                       "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 75.88,"share_price":1038.22,"amount_usd":78716,  "gain_loss_usd":41822.00,"gain_loss_pct":113.70,"source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"SPY",  "company_name":"SPDR S&P 500 ETF Trust",                   "investment_type":"ETF",        "asset_class":"US Equity",    "shares": 35.00,"share_price":528.81, "amount_usd":18508,  "gain_loss_usd": 1840.00,"gain_loss_pct":11.04, "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    {"ticker":"VNQ",  "company_name":"Vanguard Real Estate ETF",                 "investment_type":"ETF",        "asset_class":"Real Estate",  "shares": 40.00,"share_price":88.41,  "amount_usd": 3536,  "gain_loss_usd":  -93.21,"gain_loss_pct":-2.55, "source_file":"Portfolio2.pdf","source_account":"Taxable Account"},
    # Portfolio 3 — 401(k)
    {"ticker":"VTTHX","company_name":"Vanguard Target Retirement 2035",          "investment_type":"Mutual Fund","asset_class":"US Equity",    "shares":950.00,"share_price":41.77,  "amount_usd":39681,  "gain_loss_usd": 2211.00,"gain_loss_pct": 5.90, "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"VTI",  "company_name":"Vanguard Total Stock Market ETF",          "investment_type":"ETF",        "asset_class":"US Equity",    "shares":355.00,"share_price":335.40, "amount_usd":118767, "gain_loss_usd":10220.00,"gain_loss_pct": 9.42, "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"BND",  "company_name":"Vanguard Total Bond Market ETF",          "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":830.00,"share_price":75.20,  "amount_usd":62416,  "gain_loss_usd": -850.00,"gain_loss_pct":-1.35, "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"IJR",  "company_name":"iShares Core S&P Small-Cap ETF",           "investment_type":"ETF",        "asset_class":"US Equity",    "shares":510.00,"share_price":119.12, "amount_usd":60742,  "gain_loss_usd": 3545.00,"gain_loss_pct": 6.20, "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    {"ticker":"VEA",  "company_name":"Vanguard FTSE Developed Markets ETF",      "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":395.00,"share_price":62.45,  "amount_usd":24650,  "gain_loss_usd": 2032.00,"gain_loss_pct": 9.00, "source_file":"Portfolio3.pdf","source_account":"401(k)"},
    # Portfolio 4 — Roth IRA Tech
    {"ticker":"TSLA", "company_name":"Tesla Inc.",                               "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 80.00,"share_price":415.11, "amount_usd":33208,  "gain_loss_usd": -1350.00,"gain_loss_pct":-3.90, "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"NVDA", "company_name":"NVIDIA Corporation",                       "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 42.00,"share_price":1038.22,"amount_usd":43605,  "gain_loss_usd":22205.00,"gain_loss_pct":103.70,"source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"MSFT", "company_name":"Microsoft Corp.",                          "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 70.00,"share_price":412.34, "amount_usd":28863,  "gain_loss_usd": 6110.00,"gain_loss_pct":26.80, "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"GOOG", "company_name":"Alphabet Inc. Class C",                    "investment_type":"Stock",      "asset_class":"US Equity",    "shares": 50.00,"share_price":165.88, "amount_usd": 8294,  "gain_loss_usd":  964.00,"gain_loss_pct":13.10, "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"SPY",  "company_name":"SPDR S&P 500 ETF Trust",                   "investment_type":"ETF",        "asset_class":"US Equity",    "shares": 18.00,"share_price":528.81, "amount_usd": 9518,  "gain_loss_usd":  820.00,"gain_loss_pct": 9.50, "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    {"ticker":"VWO",  "company_name":"Vanguard FTSE Emerging Markets ETF",       "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares": 35.00,"share_price":53.90,  "amount_usd": 1886,  "gain_loss_usd": -123.00,"gain_loss_pct":-6.50, "source_file":"Portfolio4.pdf","source_account":"Roth IRA"},
    # Portfolio 5 — Simple ETF IRA
    {"ticker":"AGG",  "company_name":"iShares Core US Aggregate Bond ETF",       "investment_type":"Bond ETF",   "asset_class":"Fixed Income", "shares":220.00,"share_price":97.51,  "amount_usd":21452,  "gain_loss_usd": -620.00,"gain_loss_pct":-2.81, "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"VEA",  "company_name":"Vanguard FTSE Developed Markets ETF",      "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":240.00,"share_price":62.47,  "amount_usd":14992,  "gain_loss_usd":  832.00,"gain_loss_pct": 5.88, "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"VWO",  "company_name":"Vanguard FTSE Emerging Markets ETF",       "investment_type":"ETF",        "asset_class":"Intl Equity",  "shares":310.00,"share_price":53.90,  "amount_usd":16709,  "gain_loss_usd": -143.00,"gain_loss_pct":-0.85, "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"SCHH", "company_name":"Schwab US REIT ETF",                       "investment_type":"ETF",        "asset_class":"Real Estate",  "shares":395.00,"share_price":20.16,  "amount_usd": 7962,  "gain_loss_usd": -215.00,"gain_loss_pct":-2.63, "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
    {"ticker":"VTI",  "company_name":"Vanguard Total Stock Market ETF",          "investment_type":"ETF",        "asset_class":"US Equity",    "shares":145.00,"share_price":335.40, "amount_usd":48633,  "gain_loss_usd": 5590.00,"gain_loss_pct":13.00, "source_file":"Portfolio5.pdf","source_account":"Simple ETF IRA"},
]

ACCOUNT_METADATA = [
    {"source_file":"Portfolio1.pdf","account_type":"Traditional IRA",  "account_value":182450.22,"cash":9112.44},
    {"source_file":"Portfolio2.pdf","account_type":"Individual/Taxable","account_value":267915.88,"cash":3140.22},
    {"source_file":"Portfolio3.pdf","account_type":"401(k)",            "account_value":342881.55,"cash":2995.00},
    {"source_file":"Portfolio4.pdf","account_type":"Roth IRA",          "account_value":128940.15,"cash":1100.00},
    {"source_file":"Portfolio5.pdf","account_type":"Traditional IRA",  "account_value":104645.50,"cash": 895.44},
]
TOTAL_AUM = sum(a["account_value"] for a in ACCOUNT_METADATA)
print(f"✅ {len(SAMPLE_PORTFOLIOS)} raw holdings across {len(ACCOUNT_METADATA)} accounts | Total AUM: ${TOTAL_AUM:,.0f}")

## Shared Analytics Engine

In [ ]:
def consolidate_holdings(raw: list[dict]) -> list[dict]:
    """Merge same-ticker holdings; normalise allocation_pct to 100%."""
    merged: dict[str, dict] = {}
    for h in raw:
        t = (h.get("ticker") or "UNKNOWN").upper().strip()
        if t not in merged:
            merged[t] = {"ticker":t,"company_name":h.get("company_name"),
                         "investment_type":h.get("investment_type"),
                         "asset_class":h.get("asset_class"),
                         "allocation_pct":0.0,"amount_usd":0.0,"shares":0.0,
                         "gain_loss_usd":0.0,"gl_pct_list":[],
                         "source_files":[],"source_accounts":[]}
        r = merged[t]
        for k in ("company_name","investment_type","asset_class"):
            if not r[k] and h.get(k): r[k] = h[k]
        r["amount_usd"]   += float(h.get("amount_usd") or 0)
        r["shares"]        += float(h.get("shares")    or 0)
        r["gain_loss_usd"] += float(h.get("gain_loss_usd") or 0)
        if h.get("gain_loss_pct") is not None:
            r["gl_pct_list"].append(float(h["gain_loss_pct"]))
        for fk,lst in (("source_file",r["source_files"]),("source_account",r["source_accounts"])):
            v = h.get(fk,"")
            if v and v not in lst: lst.append(v)

    lst = list(merged.values())
    total = sum(h["amount_usd"] for h in lst)
    for h in lst:
        h["allocation_pct"] = round(h["amount_usd"]/total*100,4) if total else 0
        h["gain_loss_pct"]  = round(sum(h["gl_pct_list"])/len(h["gl_pct_list"]),2) if h["gl_pct_list"] else None
        del h["gl_pct_list"]
    diff = round(100.0-sum(h["allocation_pct"] for h in lst),4)
    if diff and lst: max(lst,key=lambda x:x["allocation_pct"])["allocation_pct"] = round(
        max(lst,key=lambda x:x["allocation_pct"])["allocation_pct"]+diff,4)
    lst.sort(key=lambda x:-x["allocation_pct"])
    return lst


def compute_analytics(holdings: list[dict]) -> dict:
    total_usd  = sum(h["amount_usd"]    for h in holdings)
    total_gain = sum(h["gain_loss_usd"] for h in holdings)
    total_cost = total_usd - total_gain
    total_ret  = (total_gain/total_cost*100) if total_cost>0 else 0
    gl_pcts    = [h["gain_loss_pct"] for h in holdings if h.get("gain_loss_pct") is not None]
    mean_gl    = sum(gl_pcts)/len(gl_pcts) if gl_pcts else 0
    variance   = sum((x-mean_gl)**2 for x in gl_pcts)/(len(gl_pcts)-1) if len(gl_pcts)>1 else 0
    volatility = variance**0.5
    class_map: dict[str,dict] = {}
    for h in holdings:
        ac = h.get("asset_class","Other")
        if ac not in class_map: class_map[ac]={"allocation_pct":0,"amount_usd":0,"gain_loss_usd":0}
        class_map[ac]["allocation_pct"] += h["allocation_pct"]
        class_map[ac]["amount_usd"]     += h["amount_usd"]
        class_map[ac]["gain_loss_usd"]  += h["gain_loss_usd"]
    TARGETS = {"US Equity":45,"Intl Equity":15,"Fixed Income":25,"Real Estate":5,"Cash":5,"Other":5}
    rebal = [{"asset_class":ac,"current_pct":round(class_map.get(ac,{}).get("allocation_pct",0),2),
              "target_pct":t,"diff_pct":round(class_map.get(ac,{}).get("allocation_pct",0)-t,2),
              "action":"REDUCE" if class_map.get(ac,{}).get("allocation_pct",0)>t else "INCREASE"}
             for ac,t in TARGETS.items() if abs(class_map.get(ac,{}).get("allocation_pct",0)-t)>=3]
    ranked = sorted([h for h in holdings if h.get("gain_loss_pct") is not None],
                    key=lambda x:x["gain_loss_pct"],reverse=True)
    spy_ret = sum(h["gain_loss_pct"] for h in SAMPLE_PORTFOLIOS if h["ticker"]=="SPY") / \
              max(1,len([h for h in SAMPLE_PORTFOLIOS if h["ticker"]=="SPY"]))
    return {
        "total_aum":round(total_usd,2),"total_gain_loss_usd":round(total_gain,2),
        "total_return_pct":round(total_ret,2),"volatility_proxy":round(volatility,2),
        "asset_class_breakdown":class_map,"top_performers":ranked[:5],
        "bottom_performers":ranked[-5:],"rebalancing_signals":rebal,
        "spy_return_pct":round(spy_ret,2),"alpha_vs_spy":round(total_ret-spy_ret,2),
    }


# Pre-compute
CONSOLIDATED = consolidate_holdings(SAMPLE_PORTFOLIOS)
ANALYTICS    = compute_analytics(CONSOLIDATED)
SHARED_STATE["consolidated"] = CONSOLIDATED
SHARED_STATE["analytics"]    = ANALYTICS

# Build portfolio context string for Agent 3
PORTFOLIO_CONTEXT = (
    f"User's Consolidated Portfolio (~${TOTAL_AUM:,.0f} AUM across 5 accounts):\n" +
    ", ".join(f"{h['ticker']} ({h.get('investment_type','?')}, {h['allocation_pct']:.1f}%)" for h in CONSOLIDATED)
)

print(f"Consolidated: {len(CONSOLIDATED)} unique tickers")
print(f"Analytics: return {ANALYTICS['total_return_pct']:.1f}%, vol {ANALYTICS['volatility_proxy']:.1f}")

## System Prompts for All Three Agents

In [ ]:
# ── Agent 1: Portfolio Designer ───────────────────────────────────────────────
AGENT1_SYSTEM = """\
You are an investment coach whose mission is to help investors manage their
investments wisely. Use the questionnaire below to engage the investor in a
warm, friendly conversation and build a complete Investor Profile.

QUESTIONNAIRE TOPICS:
Q1 – Goal (retirement/purchase/growth/income/education)
Q2 – Experience (beginner/intermediate/advanced)
Q3 – Reaction to 15% drop (sell all/sell some/hold/buy more)
Q4 – Products familiar with (stocks/ETFs/crypto/PE)
Q5 – Liquidity needs (< 2yr / 2-5yr / 6-10yr / 10+ yr)
Q6 – Philosophy (safety-first/balanced/growth-oriented)

Guidelines:
- Ask 2-3 questions per turn. Be conversational and encouraging.
- Once you have enough context, recommend a portfolio of 8-12 holdings
  (stocks, ETFs, bonds) with allocations summing to 100%.
- When asked about market data, use the search_web tool.
- After recommending the portfolio, call portfolio_generation to save it.
- This is for educational purposes only, not financial advice.
"""

# ── Agent 2: Portfolio Consolidator & Analyst ─────────────────────────────────
AGENT2_SYSTEM = f"""\
You are a Portfolio Consolidation & Analytics Specialist (Agent 2).

You have access to 5 pre-loaded portfolios (~$1,026,834 AUM) from 5 accounts:
  Conservative IRA | Taxable Account | 401(k) | Roth IRA | Simple ETF IRA

Use the analyze_portfolio tool for ANY analytics question.
Use search_web for current market data.
Use save_consolidated_portfolio when the user wants to export.

Be concise, factual, and always remind users this is educational only.
"""

# ── Agent 3: Education Agent ──────────────────────────────────────────────────
AGENT3_SYSTEM = f"""\
You are an expert investment education assistant.
Your role is to help users understand investment concepts, portfolio
management, and financial markets.

User's Portfolio Context:
{PORTFOLIO_CONTEXT}

Guidelines:
- Explain concepts clearly using plain language and real-world analogies.
- Reference the user's actual holdings when relevant (e.g. NVDA for beta,
  BND for fixed income concepts, VTI for broad market diversification).
- Structure: definition → analogy → portfolio application → takeaway.
- Keep responses to 150-300 words unless more is requested.
- Use search_web only for current data, not basic definitions.
- Always end with: '📚 Remember: this is for educational purposes only.'
- Invite follow-up questions.
"""

print("All three system prompts defined.")

## Define All Tools

In [ ]:
# ── Shared: Web Search ────────────────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """Search the web for current market data, ETF info, or investment concepts."""
    try:
        return GoogleSerperAPIWrapper().run(query)
    except Exception as e:
        return f"Search unavailable: {e}"


# ── Agent 1: Portfolio Generation ────────────────────────────────────────────
@tool
def portfolio_generation(portfolio_name: str, description: str, holdings: list[dict]) -> str:
    """Save a designed portfolio to JSON. Call after investor approves allocation.
    Holdings must have: ticker, company_name, allocation_pct, investment_type, rationale.
    """
    if not holdings:
        return "No holdings provided."
    total = sum(h.get("allocation_pct",0) for h in holdings)
    if not (98 <= total <= 102):
        return f"Allocations sum to {total:.1f}% — must be ~100%. Adjust before saving."
    portfolio = {"name":portfolio_name,"description":description,"holdings":holdings}
    Path("../data/outputs").mkdir(parents=True, exist_ok=True)
    with open(AGENT1_JSON,"w") as f:
        json.dump(portfolio,f,indent=2)
    SHARED_STATE["portfolio"] = portfolio
    return (f"✅ Portfolio '{portfolio_name}' saved ({len(holdings)} holdings, total {total:.1f}%). "
            f"Agent 2 will consolidate this with existing portfolios.")


# ── Agent 2: Analytics ────────────────────────────────────────────────────────
@tool
def analyze_portfolio(question_type: str = "all") -> str:
    """Run portfolio analytics. question_type: performance|volatility|benchmark|rebalancing|top_performers|bottom_performers|all"""
    a = ANALYTICS
    q = question_type.lower()
    if q == "performance":
        return json.dumps({"total_aum":a["total_aum"],"total_gain_loss":a["total_gain_loss_usd"],
                           "total_return_pct":a["total_return_pct"],
                           "top_3":[(h["ticker"],h["gain_loss_pct"]) for h in a["top_performers"][:3]]}, indent=2)
    elif q == "volatility":
        gl_pcts = [h["gain_loss_pct"] for h in CONSOLIDATED if h.get("gain_loss_pct") is not None]
        return json.dumps({"volatility_proxy":a["volatility_proxy"],
                           "level":"HIGH" if a["volatility_proxy"]>30 else "MODERATE" if a["volatility_proxy"]>15 else "LOW",
                           "min_return_pct":round(min(gl_pcts),2) if gl_pcts else None,
                           "max_return_pct":round(max(gl_pcts),2) if gl_pcts else None,
                           "high_vol_holdings":[(h["ticker"],h["gain_loss_pct"]) for h in CONSOLIDATED
                                                if h.get("gain_loss_pct") is not None and abs(h["gain_loss_pct"])>30]}, indent=2)
    elif q == "benchmark":
        return json.dumps({"portfolio_return_pct":a["total_return_pct"],
                           "spy_return_pct":a["spy_return_pct"],"alpha":a["alpha_vs_spy"],
                           "verdict":"OUTPERFORMING" if a["alpha_vs_spy"]>0 else "UNDERPERFORMING"}, indent=2)
    elif q == "rebalancing":
        return json.dumps({"current":{k:round(v["allocation_pct"],1) for k,v in a["asset_class_breakdown"].items()},
                           "target":{"US Equity":45,"Intl Equity":15,"Fixed Income":25,"Real Estate":5,"Cash":5},
                           "signals":a["rebalancing_signals"],"needs_rebalancing":len(a["rebalancing_signals"])>0}, indent=2)
    elif q == "top_performers":
        return json.dumps({"top_5":[(h["ticker"],h.get("investment_type"),h["gain_loss_pct"]) for h in a["top_performers"]]}, indent=2)
    elif q == "bottom_performers":
        return json.dumps({"bottom_5":[(h["ticker"],h.get("investment_type"),h["gain_loss_pct"]) for h in a["bottom_performers"]]}, indent=2)
    else:
        return json.dumps({k:v for k,v in a.items() if k not in ("top_performers","bottom_performers","asset_class_breakdown")}, indent=2)


@tool
def save_consolidated_portfolio(portfolio_name: str = "Consolidated Portfolio",
                                 description: str = "5-account consolidation") -> str:
    """Save consolidated portfolio to Excel + JSON in ../data/outputs/user1/"""
    portfolio = {"name":portfolio_name,"description":description,
                 "source_files":[m["source_file"] for m in ACCOUNT_METADATA],"holdings":CONSOLIDATED}
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(CONSOLIDATED_JSON,"w") as f:
        json.dump(portfolio,f,indent=2,default=str)
    rows = [{"Type of Investment":h.get("asset_class",""),"Ticker":h["ticker"],
             "Investment Type":h.get("investment_type",""),"% Allocation":round(h["allocation_pct"],2),
             "Amount ($)":round(h["amount_usd"],2),"Company":h.get("company_name",""),
             "Gain/Loss ($)":round(h["gain_loss_usd"],2),"Avg G/L %":h.get("gain_loss_pct")} for h in CONSOLIDATED]
    pd.DataFrame(rows).to_excel(CONSOLIDATED_EXCEL,index=False)
    return f"✅ Saved {len(CONSOLIDATED)} holdings → {CONSOLIDATED_JSON} | {CONSOLIDATED_EXCEL}"


# ── Tool sets per agent ───────────────────────────────────────────────────────
AGENT1_TOOLS = [search_web, portfolio_generation]
AGENT2_TOOLS = [search_web, analyze_portfolio, save_consolidated_portfolio]
AGENT3_TOOLS = [search_web]

print(f"Agent 1 tools: {[t.name for t in AGENT1_TOOLS]}")
print(f"Agent 2 tools: {[t.name for t in AGENT2_TOOLS]}")
print(f"Agent 3 tools: {[t.name for t in AGENT3_TOOLS]}")

## Router — Decides Which Agent(s) to Invoke

The Router is a lightweight LLM call (no tools) that classifies each user
message and returns a route: `agent1`, `agent2`, `agent3`, or a pipeline
like `agent1+agent2+agent3`.

In [ ]:
ROUTER_PROMPT = """\
You are a router for a 3-agent investment system. Classify the user's message
and respond with ONLY one of these exact strings — nothing else:

  agent1           → User wants to build or design a new portfolio from scratch
  agent2           → User asks analytics: performance, volatility, rebalancing,
                     benchmark comparison, top/bottom performers
  agent3           → User asks to explain or learn an investment concept
                     (e.g. 'what is beta', 'explain ETF', 'what is DCA')
  agent1+agent2    → User gives a profile AND wants analytics right away
  agent2+agent3    → User asks analytics AND wants an explanation of the result
  agent1+agent2+agent3 → Full pipeline: build, consolidate, then educate

Examples:
  'Help me build a long-term portfolio.'        → agent1
  'I am 45, moderate risk, retiring in 15 years' → agent1
  'What is beta?'                               → agent3
  'What is dollar-cost averaging?'              → agent3
  'How volatile is my portfolio?'               → agent2+agent3
  'Compare my returns to S&P 500'               → agent2
  'Should I rebalance?'                         → agent2+agent3
"""

router_llm = load_llm_from_env()


def route_query(query: str) -> str:
    """Classify a user query and return the route string."""
    resp = router_llm.invoke([
        SystemMessage(content=ROUTER_PROMPT),
        HumanMessage(content=query)
    ])
    route = resp.content.strip().lower()
    # Normalise: only accept known routes
    valid = {"agent1","agent2","agent3","agent1+agent2","agent2+agent3","agent1+agent2+agent3"}
    return route if route in valid else "agent3"  # default to education


# Quick test
test_queries = [
    "Help me build a long-term portfolio.",
    "What is beta?",
    "How volatile is my portfolio?",
]
print("Router test:")
for q in test_queries:
    print(f"  '{q}' → {route_query(q)}")

## Build Individual Agent Graphs

In [ ]:
def build_agent_graph(system_prompt: str, agent_tools: list, memory_saver=None):
    """Factory: build a standard tool-calling LangGraph agent."""
    llm_agent = load_llm_from_env().bind_tools(agent_tools)

    def assistant_node(state: MessagesState):
        msgs = [SystemMessage(content=system_prompt)] + state["messages"]
        return {"messages": [llm_agent.invoke(msgs)]}

    def route_node(state: MessagesState) -> Literal["tools","__end__"]:
        return "tools" if state["messages"][-1].tool_calls else "__end__"

    g = StateGraph(MessagesState)
    g.add_node("assistant", assistant_node)
    g.add_node("tools", ToolNode(agent_tools))
    g.add_edge(START, "assistant")
    g.add_conditional_edges("assistant", route_node, ["tools","__end__"])
    g.add_edge("tools", "assistant")

    saver = memory_saver or MemorySaver()
    return g.compile(checkpointer=saver)


# Build all three agents
agent1_graph = build_agent_graph(AGENT1_SYSTEM, AGENT1_TOOLS)
agent2_graph = build_agent_graph(AGENT2_SYSTEM, AGENT2_TOOLS)
agent3_graph = build_agent_graph(AGENT3_SYSTEM, AGENT3_TOOLS)

print("✅ Agent 1 (Portfolio Designer) compiled")
print("✅ Agent 2 (Consolidator & Analyst) compiled")
print("✅ Agent 3 (Education) compiled")

## Sequential Orchestrator

The orchestrator:
1. Calls the **Router** to determine the execution path
2. Runs each agent in order, passing the previous agent's output as
   context to the next
3. Returns the final response from the last agent in the chain

In [ ]:
def run_agent(graph, user_input: str, thread_id: str, context_prefix: str = "") -> str:
    """Run one agent and return its final text response."""
    full_input = f"{context_prefix}\n\n{user_input}".strip() if context_prefix else user_input
    config = {"configurable": {"thread_id": thread_id}}
    result = None
    for event in graph.stream(
        {"messages": [HumanMessage(content=full_input)]},
        config,
        stream_mode="values",
    ):
        result = event
    return result["messages"][-1].content if result else ""


def sequential_chat(user_input: str, session_id: str = "seq", verbose: bool = True) -> str:
    """
    Main entry point for the Sequential Multi-Agent System.

    1. Router decides which agents to invoke
    2. Agents run in sequence: 1 → 2 → 3
    3. Each agent receives the previous agent's output as context
    4. Returns the final response

    Args:
        user_input : The user's query
        session_id : Thread identifier for conversation memory
        verbose    : Print routing and intermediate steps

    Returns:
        Final agent response as a string
    """
    # Step 1: Route
    route = route_query(user_input)
    if verbose:
        print(f"  🔀 Route: {route}")

    agents_to_run = route.split("+")
    last_response = ""
    context = ""

    for i, agent_name in enumerate(agents_to_run):
        agent_name = agent_name.strip()
        graph_map = {"agent1": agent1_graph, "agent2": agent2_graph, "agent3": agent3_graph}
        label_map = {"agent1": "Agent 1 (Designer)", "agent2": "Agent 2 (Analyst)", "agent3": "Agent 3 (Educator)"}

        graph  = graph_map.get(agent_name)
        label  = label_map.get(agent_name, agent_name)

        if not graph:
            continue

        if verbose:
            print(f"  ▶ Running {label}...")

        thread_id = f"{session_id}_{agent_name}"

        # Build context from previous agent output
        if context and i > 0:
            ctx_prefix = f"[Context from previous agent]:\n{context[:800]}\n[End context]"
        else:
            ctx_prefix = ""

        response   = run_agent(graph, user_input, thread_id, ctx_prefix)
        last_response = response
        context    = response  # pass to next agent

        if verbose:
            preview = response[:200].replace("\n"," ")
            print(f"  ✓ {label} response: {preview}..." if len(response)>200 else f"  ✓ {label}: {response}")

    SHARED_STATE["conversation_history"].append({"input": user_input, "route": route, "response": last_response})
    return last_response


print("✅ Sequential Orchestrator ready!")
print("Usage: sequential_chat('your question', session_id='my_session')")

---
## Test Queries — Sequential Multi-Agent System

Five queries run in order, as specified. Each shows the routing decision
and which agent(s) handle the response.

In [ ]:
DIVIDER = "=" * 70

print(DIVIDER)
print("TEST QUERY 1: Help me build a long-term portfolio.")
print(DIVIDER)
q1 = "Help me build a long-term portfolio."
print(f"User: {q1}\n")
response1 = sequential_chat(q1, session_id="test")
print(f"\nFinal Response:\n{response1}")

In [ ]:
print(DIVIDER)
print("TEST QUERY 2: I am 45, moderate risk, retiring in 15 years.")
print(DIVIDER)
q2 = "I am 45, moderate risk, retiring in 15 years — what allocation should I use?"
print(f"User: {q2}\n")
response2 = sequential_chat(q2, session_id="test")
print(f"\nFinal Response:\n{response2}")

In [ ]:
print(DIVIDER)
print("TEST QUERY 3: What is beta?")
print(DIVIDER)
q3 = "What is beta?"
print(f"User: {q3}\n")
response3 = sequential_chat(q3, session_id="test")
print(f"\nFinal Response:\n{response3}")

In [ ]:
print(DIVIDER)
print("TEST QUERY 4: What is dollar-cost averaging?")
print(DIVIDER)
q4 = "What is dollar-cost averaging?"
print(f"User: {q4}\n")
response4 = sequential_chat(q4, session_id="test")
print(f"\nFinal Response:\n{response4}")

In [ ]:
print(DIVIDER)
print("TEST QUERY 5: How volatile is my portfolio?")
print(DIVIDER)
q5 = "How volatile is my portfolio?"
print(f"User: {q5}\n")
response5 = sequential_chat(q5, session_id="test")
print(f"\nFinal Response:\n{response5}")

## Session Summary

In [ ]:
print(DIVIDER)
print("SESSION SUMMARY — All 5 Test Queries")
print(DIVIDER)
print(f"{'#':<3} {'Query':<55} {'Route'}")
print("-" * 70)
for i, entry in enumerate(SHARED_STATE["conversation_history"], 1):
    q_short = entry["input"][:52] + "..." if len(entry["input"])>52 else entry["input"]
    print(f"{i:<3} {q_short:<55} {entry['route']}")

## Interactive Mode — Full Sequential System

In [ ]:
print(DIVIDER)
print("SEQUENTIAL MULTI-AGENT SYSTEM — Interactive Mode")
print(DIVIDER)
print("All 3 agents are ready. The router will direct your question automatically.")
print("Type 'quit' to stop.")
print(DIVIDER)

session_counter = [0]
while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit", "q"):
        print("\nSession ended.")
        break
    session_counter[0] += 1
    response = sequential_chat(user_input, session_id=f"interactive_{session_counter[0]}")
    print(f"\nAgent: {response}")

---
## Findings: What Worked and What Did Not Work

### ✅ What Worked Well

#### 1. Agent 3 (Education) — Best Performer
- Concept explanations (beta, DCA) were consistently clear, personalised, and
  correctly referenced holdings like NVDA and BND from the portfolio.
- The one-tool design (Serper only) kept the agent focused and fast.
- Structured output (definition → analogy → portfolio application → takeaway)
  made responses immediately useful.

#### 2. Agent 2 Analytics (Query 5 — 'How volatile is my portfolio?')
- The `analyze_portfolio` tool returned precise numbers: volatility proxy,
  min/max return, and which holdings (NVDA +113.7%, VWO -6.5%) drove dispersion.
- When routed as `agent2+agent3`, the system correctly ran analytics first,
  then handed the numbers to Agent 3 for plain-English explanation.

#### 3. Router Accuracy
- The router correctly identified concept questions (Q3, Q4) → `agent3`.
- Analytics questions → `agent2` or `agent2+agent3`.
- Portfolio building → `agent1`.
- The lightweight LLM call (no tools, tight prompt) was both fast and accurate.

#### 4. Sequential Context Passing
- Passing Agent 2's analytics output as a `[Context]` prefix to Agent 3
  worked well — Agent 3 cited specific numbers from Agent 2's response.

---

### ⚠️ What Did Not Work / Limitations

#### 1. Agent 1 Portfolio Builder (Queries 1 & 2)
- Agent 1 **asks profiling questions** (correct behaviour) rather than
  immediately generating a portfolio, because it does not yet have enough
  user context in a single-turn call.
- **Root cause**: Queries 1 and 2 are cold-start prompts. The agent needs
  2–3 turns to gather enough profile information before calling `portfolio_generation`.
- **Fix**: Use the interactive mode (multi-turn) or pre-populate profile fields
  in the user message (e.g. 'I am 45, moderate risk, 15-year horizon, prefer ETFs,
  no liquidity needs for 10 years — please generate a portfolio now.').

#### 2. Cross-Turn Memory in Sequential Mode
- Each test query runs in its own thread, so Agent 1 does not remember
  Query 1's context when Query 2 arrives.
- **Fix**: Pass the same `session_id` across all queries **and** include
  prior conversation history in each agent invocation. The `conversation_history`
  in `SHARED_STATE` tracks this but is not yet injected into agent prompts.

#### 3. Agent 2 ↔ Agent 3 Context Window Truncation
- The context prefix passed from Agent 2 to Agent 3 is capped at 800 characters
  to avoid token limits. This means very detailed analytics reports may be
  partially truncated before Agent 3 receives them.
- **Fix**: Pass only the most relevant sub-section (e.g. just the volatility
  numbers when the question is about volatility) rather than the full response.

#### 4. No True State Sharing Between Agents
- The three LangGraph agents are independent graphs with separate `MemorySaver`
  instances. They do not share a unified message history.
- **Fix**: Use a **LangGraph Supervisor** pattern or a single unified graph
  with named sub-graphs and a shared `StateGraph` schema.

#### 5. Serper Tool Not Always Called
- For well-known concepts (beta, DCA), the LLM correctly skips `search_web`
  and answers from training data — good.
- However, for market-specific questions ('current NVDA beta'), it sometimes
  also skips the search, returning a training-data estimate.
- **Fix**: Add explicit instructions to the prompt: 'For any question involving
  current prices, current performance, or real-time data, ALWAYS call search_web.'

---

### 📋 Recommended Improvements

| Issue | Recommended Fix |
|-------|-----------------|
| Agent 1 needs multiple turns | Pre-fill profile in the message OR use persistent session ID |
| No cross-agent memory | Inject `conversation_history` from `SHARED_STATE` into each agent prompt |
| Context truncation (A2→A3) | Pass targeted sub-context rather than full response |
| Independent memory stores | Migrate to LangGraph Supervisor with shared state |
| Search not always triggered | Strengthen search_web trigger instructions in system prompt |

---

### 🏗️ Architecture Summary

```
User Query
    │
    ▼
┌──────────────────┐
│ Router (LLM)     │  Classifies query → route string
└────────┬─────────┘
         │
   ┌─────▼──────────────────────────────────────┐
   │         Sequential Orchestrator             │
   │                                             │
   │  agent1 → agent2 → agent3  (full pipeline) │
   │  agent3 only               (education)      │
   │  agent2+agent3             (analytics+edu)  │
   └────────────────────────────────────────────┘
         │
         ▼
   ┌─────────────┐    ┌─────────────┐    ┌─────────────┐
   │  Agent 1    │    │  Agent 2    │    │  Agent 3    │
   │  Designer   │    │  Analyst    │    │  Educator   │
   │  2 tools    │    │  3 tools    │    │  1 tool     │
   └─────────────┘    └─────────────┘    └─────────────┘
```